# 📚 Notebook 01 — Market Data Foundations

**Phase 1: Foundations** · Prerequisites: Basic Python

---

## 🎯 Learning Objectives

After this notebook, you will be able to:

1. Explain what **OHLCV** data is and read a candlestick chart
2. **Fetch historical klines** from the Binance public API using Python
3. Build a **structured dataclass** (`BinanceKline`) for financial records
4. Handle **API pagination** to retrieve long histories (1000-row limit per page)
5. **Store and query** kline data in a local SQLite database
6. Plot professional **candlestick charts** with matplotlib

In [ ]:
# ── Environment Setup (run once per session) ──
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone, timedelta

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
print("✅ Setup complete")

---

## 📐 Section 1: What Is OHLCV Data?

Every trading system begins with **price data**. The universal format is **OHLCV** — five numbers that summarize all trading activity in one time period:

| Field | Meaning | Example |
|---|---|---|
| **O**pen | Price at the start of the period | $67,450.00 |
| **H**igh | Highest price during the period | $68,200.00 |
| **L**ow | Lowest price during the period | $67,100.00 |
| **C**lose | Price at the end of the period | $67,800.00 |
| **V**olume | Total units traded during the period | 1,234.56 BTC |

### Candlestick Anatomy

```
     ┃       ← Upper wick (High)
   ┌─┴─┐
   │   │     ← Body (Open → Close)
   │   │        Green if Close > Open (price went up)
   │   │        Red if Close < Open (price went down)
   └─┬─┘
     ┃       ← Lower wick (Low)
```

Each candle represents one **time interval** (1 minute, 1 hour, 1 day, etc.).

### Why OHLCV?

- **Compact**: Summarizes thousands of individual trades into 5 numbers
- **Universal**: Every exchange, every asset class uses this format
- **Sufficient**: Most quantitative strategies only need OHLCV data
- **Volume matters**: Volume confirms price moves — a breakout on high volume is more reliable

---

## 💻 Section 2: Fetching Data from Binance

Binance offers a **free public API** for historical kline (candlestick) data. No API key needed.

### The Endpoint

```
GET https://api.binance.com/api/v3/klines
    ?symbol=BTCUSDT
    &interval=1h
    &limit=1000
    &startTime=<ms_timestamp>
    &endTime=<ms_timestamp>
```

Each row returns **13 values** as a JSON array:

| Index | Field | Type |
|---|---|---|
| 0 | Open time (ms) | int |
| 1 | Open price | string |
| 2 | High price | string |
| 3 | Low price | string |
| 4 | Close price | string |
| 5 | Volume (base asset) | string |
| 6 | Close time (ms) | int |
| 7 | Quote asset volume | string |
| 8 | Number of trades | int |
| 9 | Taker buy base volume | string |
| 10 | Taker buy quote volume | string |

Let's fetch some data:

In [ ]:
# ── Raw API call: fetch 100 hourly BTC candles ──
import requests

url = "https://api.binance.com/api/v3/klines"
params = {
    "symbol": "BTCUSDT",
    "interval": "1h",
    "limit": 100,
}

response = requests.get(url, params=params, timeout=10)
response.raise_for_status()
raw_klines = response.json()

print(f"Fetched {len(raw_klines)} klines")
print(f"First kline (raw): {raw_klines[0][:5]}...")  # Show first 5 fields

### Problem: Raw Data Is Messy

The API returns **arrays of mixed types** — timestamps as ints, prices as strings, volumes as strings. We need a clean, typed structure.

### Solution: The `BinanceKline` Dataclass

Our production bot uses a **frozen dataclass** — an immutable named record with proper types. Let's build one from scratch, then compare to the production version.

In [ ]:
# ── Building a BinanceKline dataclass from scratch ──
from dataclasses import dataclass
from typing import Any

@dataclass(frozen=True, slots=True)
class Kline:
    """Normalized OHLCV record from Binance.
    
    frozen=True  → immutable (can't accidentally modify data)
    slots=True   → memory-efficient (no __dict__)
    """
    symbol: str
    interval: str
    open_time_ms: int      # Unix timestamp in milliseconds
    close_time_ms: int
    open: float
    high: float
    low: float
    close: float
    volume: float          # Base asset volume (e.g., BTC)
    quote_volume: float    # Quote asset volume (e.g., USDT)
    trade_count: int

    @classmethod
    def from_api_row(cls, symbol: str, interval: str, row: list[Any]) -> "Kline":
        """Parse one raw API row into a typed Kline record."""
        return cls(
            symbol=symbol,
            interval=interval,
            open_time_ms=int(row[0]),
            open=float(row[1]),
            high=float(row[2]),
            low=float(row[3]),
            close=float(row[4]),
            volume=float(row[5]),
            close_time_ms=int(row[6]),
            quote_volume=float(row[7]),
            trade_count=int(row[8]),
        )

# Parse the raw data into typed records
klines = [Kline.from_api_row("BTCUSDT", "1h", row) for row in raw_klines]

print(f"\nParsed {len(klines)} klines")
print(f"First: open={klines[0].open:,.2f}, close={klines[0].close:,.2f}, volume={klines[0].volume:,.4f}")
print(f"Last:  open={klines[-1].open:,.2f}, close={klines[-1].close:,.2f}")

### 💻 Production Code Comparison

Our simplified `Kline` is very close to the production `BinanceKline` in `bot/data/binance_fetcher.py`. The production version also includes:

- `taker_buy_base_volume` and `taker_buy_quote_volume` (aggressive buyer tracking)
- `normalize_binance_symbol()` to handle symbol format differences (e.g., `BTCUSD` → `BTCUSDT`)
- Integration with the `tenacity` retry library for resilient API calls

Let's look at the production fetcher:

In [ ]:
# ── Production BinanceFetcher from the bot ──
from bot.data.binance_fetcher import BinanceFetcher, BinanceKline, normalize_binance_symbol

# Key features of the production fetcher:
print("Symbol normalization examples:")
print(f"  BTCUSD  → {normalize_binance_symbol('BTCUSD')}")
print(f"  ETHUSDT → {normalize_binance_symbol('ETHUSDT')}")
print(f"  SOL/USD → {normalize_binance_symbol('SOL/USD')}")

print(f"\nSupported intervals: {list(BinanceFetcher.interval_to_milliseconds.__func__.__code__.co_consts)}")

# The production fetcher wraps requests with:
# - @retry(stop=stop_after_attempt(3), wait=wait_exponential(...))
# - Handles HTTP 429 (rate limit) and 5xx (server error) as retryable
# - Raises BinanceApiError for non-retryable failures

---

## 📐 Section 3: API Pagination — Getting Long Histories

Binance limits each request to **1000 rows**. For 90 days of hourly data, we need:

$$\text{Total candles} = 90 \times 24 = 2{,}160 \quad \Rightarrow \quad \lceil 2{,}160 / 1{,}000 \rceil = 3 \text{ pages}$$

### Pagination Strategy

```
Page 1: startTime = T₀         → get 1000 klines → last_open_time = T₉₉₉
Page 2: startTime = T₉₉₉ + 1h → get 1000 klines → last_open_time = T₁₉₉₉
Page 3: startTime = T₁₉₉₉ + 1h → get 160 klines  → DONE (fewer than 1000)
```

The key insight: after each page, advance `startTime` to `last_open_time + interval_ms`.

In [ ]:
# ── Building a paginated fetcher from scratch ──

def fetch_all_klines(
    symbol: str,
    interval: str = "1h",
    days: int = 30,
    limit: int = 1000,
) -> list[Kline]:
    """Fetch all klines for a symbol by paging through the Binance API."""
    # Calculate time range
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(days=days)
    
    # Convert to milliseconds (Binance uses ms timestamps)
    cursor_ms = int(start_time.timestamp() * 1000)
    end_ms = int(end_time.timestamp() * 1000)
    
    # Interval width in ms (for advancing the cursor)
    interval_ms = {"1h": 3_600_000, "1d": 86_400_000, "4h": 14_400_000}[interval]
    
    all_klines: list[Kline] = []
    page = 0
    
    while cursor_ms < end_ms:
        page += 1
        response = requests.get(
            "https://api.binance.com/api/v3/klines",
            params={
                "symbol": symbol,
                "interval": interval,
                "startTime": cursor_ms,
                "endTime": end_ms,
                "limit": limit,
            },
            timeout=10,
        )
        response.raise_for_status()
        rows = response.json()
        
        if not rows:
            break
        
        page_klines = [Kline.from_api_row(symbol, interval, r) for r in rows]
        all_klines.extend(page_klines)
        
        # Advance cursor past the last kline
        last_open = page_klines[-1].open_time_ms
        cursor_ms = last_open + interval_ms
        
        print(f"  Page {page}: fetched {len(rows)} klines (total: {len(all_klines)})")
        
        if len(rows) < limit:
            break  # Last page
    
    return all_klines

# Fetch 30 days of hourly BTC data
print("Fetching BTCUSDT hourly klines (30 days)...")
btc_klines = fetch_all_klines("BTCUSDT", "1h", days=30)
print(f"\n✅ Total: {len(btc_klines)} klines")

---

## 📊 Section 4: Visualizing Candlestick Data

In [ ]:
# ── Convert to DataFrame for analysis and plotting ──

def klines_to_dataframe(klines: list[Kline]) -> pd.DataFrame:
    """Convert a list of Kline records into a pandas DataFrame."""
    records = [
        {
            "timestamp": pd.Timestamp(k.open_time_ms, unit="ms", tz="UTC"),
            "open": k.open,
            "high": k.high,
            "low": k.low,
            "close": k.close,
            "volume": k.volume,
            "quote_volume": k.quote_volume,
            "trade_count": k.trade_count,
        }
        for k in klines
    ]
    df = pd.DataFrame(records).set_index("timestamp")
    return df

btc_df = klines_to_dataframe(btc_klines)
print(f"DataFrame shape: {btc_df.shape}")
btc_df.tail()

In [ ]:
# ── Professional candlestick chart ──

def plot_candlestick(df: pd.DataFrame, title: str = "BTC/USDT", last_n: int = 120):
    """Plot an OHLCV candlestick chart with volume bars."""
    data = df.tail(last_n).copy()
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), height_ratios=[3, 1], sharex=True)
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # Candlestick body
    colors = ['#26a69a' if c >= o else '#ef5350' for o, c in zip(data['open'], data['close'])]
    
    x = range(len(data))
    
    # Wicks (high-low lines)
    for i, (idx, row) in enumerate(data.iterrows()):
        color = colors[i]
        ax1.plot([i, i], [row['low'], row['high']], color=color, linewidth=0.8)
    
    # Bodies
    body_width = 0.6
    for i, (idx, row) in enumerate(data.iterrows()):
        color = colors[i]
        bottom = min(row['open'], row['close'])
        height = abs(row['close'] - row['open'])
        ax1.bar(i, height, bottom=bottom, width=body_width, color=color, edgecolor=color)
    
    ax1.set_ylabel('Price (USDT)', fontsize=12)
    ax1.grid(True, alpha=0.3)
    
    # Volume bars
    ax2.bar(x, data['volume'], color=colors, alpha=0.7, width=body_width)
    ax2.set_ylabel('Volume (BTC)', fontsize=12)
    ax2.grid(True, alpha=0.3)
    
    # X-axis labels (show every Nth timestamp)
    step = max(1, len(data) // 10)
    ax2.set_xticks(range(0, len(data), step))
    ax2.set_xticklabels(
        [data.index[i].strftime('%m/%d %H:%M') for i in range(0, len(data), step)],
        rotation=45, ha='right'
    )
    
    plt.tight_layout()
    plt.show()

plot_candlestick(btc_df, "BTC/USDT — Hourly Candlesticks (Last 5 Days)", last_n=120)

---

## 💻 Section 5: Storing Data in SQLite

Fetching from Binance every time is **slow** and hits rate limits. The production bot caches data in a **SQLite database** — a lightweight file-based database perfect for single-user applications.

### Why SQLite?

- **Zero setup**: No server, no credentials — just a file
- **ACID transactions**: Data integrity even if the process crashes
- **Fast queries**: Indexed lookups in microseconds
- **Production-proven**: Used in the actual trading bot for kline caching

Let's build a simplified version of the production `BinanceHistoryStore`:

In [ ]:
# ── Building a SQLite kline store from scratch ──
import sqlite3
import tempfile

class KlineStore:
    """SQLite-backed storage for historical kline data.
    
    Mirrors the production BinanceHistoryStore pattern:
    - Upsert (insert or update) to handle duplicate fetches
    - Compound primary key: (symbol, interval, open_time_ms)
    - Indexed for fast time-range queries
    """
    
    def __init__(self, db_path: str):
        self.db_path = db_path
        self._initialize()
    
    def _initialize(self):
        """Create the klines table and index if they don't exist."""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS klines (
                    symbol TEXT NOT NULL,
                    interval TEXT NOT NULL,
                    open_time_ms INTEGER NOT NULL,
                    close_time_ms INTEGER NOT NULL,
                    open REAL NOT NULL,
                    high REAL NOT NULL,
                    low REAL NOT NULL,
                    close REAL NOT NULL,
                    volume REAL NOT NULL,
                    quote_volume REAL NOT NULL,
                    trade_count INTEGER NOT NULL,
                    PRIMARY KEY (symbol, interval, open_time_ms)
                )
            """)
            conn.execute("""
                CREATE INDEX IF NOT EXISTS idx_klines_lookup
                ON klines (interval, symbol, open_time_ms)
            """)
    
    def upsert(self, klines: list[Kline]) -> int:
        """Insert klines, updating existing rows on conflict."""
        if not klines:
            return 0
        with sqlite3.connect(self.db_path) as conn:
            conn.executemany("""
                INSERT INTO klines (symbol, interval, open_time_ms, close_time_ms,
                                    open, high, low, close, volume, quote_volume, trade_count)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                ON CONFLICT(symbol, interval, open_time_ms) DO UPDATE SET
                    close = excluded.close,
                    high = excluded.high,
                    low = excluded.low,
                    volume = excluded.volume,
                    quote_volume = excluded.quote_volume
            """, [
                (k.symbol, k.interval, k.open_time_ms, k.close_time_ms,
                 k.open, k.high, k.low, k.close, k.volume, k.quote_volume, k.trade_count)
                for k in klines
            ])
        return len(klines)
    
    def query(self, symbol: str, interval: str, limit: int = 500) -> pd.DataFrame:
        """Load the most recent klines as a DataFrame."""
        with sqlite3.connect(self.db_path) as conn:
            df = pd.read_sql_query(
                "SELECT * FROM klines WHERE symbol = ? AND interval = ? ORDER BY open_time_ms DESC LIMIT ?",
                conn, params=(symbol, interval, limit)
            )
        if not df.empty:
            df['timestamp'] = pd.to_datetime(df['open_time_ms'], unit='ms', utc=True)
            df = df.set_index('timestamp').sort_index()
        return df
    
    def count(self, symbol: str, interval: str) -> int:
        """Count stored klines for a symbol/interval pair."""
        with sqlite3.connect(self.db_path) as conn:
            result = conn.execute(
                "SELECT COUNT(*) FROM klines WHERE symbol = ? AND interval = ?",
                (symbol, interval)
            ).fetchone()
        return result[0]

# Store the data we fetched earlier
db_path = Path(tempfile.gettempdir()) / "quant_course_klines.db"
store = KlineStore(str(db_path))

stored = store.upsert(btc_klines)
print(f"✅ Stored {stored} klines in {db_path}")
print(f"   Database has {store.count('BTCUSDT', '1h')} BTCUSDT hourly klines")

In [ ]:
# ── Query data back from the store ──
cached_df = store.query("BTCUSDT", "1h", limit=100)
print(f"Retrieved {len(cached_df)} klines from SQLite")
cached_df.tail(3)

### 💻 Production Store Comparison

The production `BinanceHistoryStore` in `bot/data/binance_history_store.py` follows the exact same pattern:

- Same `ON CONFLICT ... DO UPDATE` upsert strategy
- Same compound primary key `(symbol, interval, open_time_ms)`
- Additional `get_time_range()` method to check what's already cached
- Uses `sqlite3.Row` factory for dict-like row access

---

## 🔬 Section 6: Interactive Exercises

### Exercise 1: Fetch ETH Data 🔬

Fetch 30 days of hourly ETHUSDT klines, store them in the same database, and plot a candlestick chart.

In [ ]:
# ── Exercise 1: Your code here ──
# 1. Call fetch_all_klines("ETHUSDT", "1h", days=30)
# 2. Store the result with store.upsert()
# 3. Query it back and plot with plot_candlestick()

# YOUR CODE HERE

In [ ]:
# ── Exercise 1: Solution (expand to check) ──
# eth_klines = fetch_all_klines("ETHUSDT", "1h", days=30)
# store.upsert(eth_klines)
# eth_df = klines_to_dataframe(eth_klines)
# plot_candlestick(eth_df, "ETH/USDT — Hourly Candlesticks", last_n=120)

### Exercise 2: Daily vs Hourly ⭐

Fetch 90 days of **daily** (`1d`) BTCUSDT klines. Compare the daily chart to the hourly chart. How does the choice of interval affect what patterns you can see?

In [ ]:
# ── Exercise 2: Your code here ──

# YOUR CODE HERE

### Exercise 3: Volume Analysis ⭐

Plot the **quote volume** (USDT traded) over time. Can you spot days with unusually high volume? What happened on those days?

In [ ]:
# ── Exercise 3: Your code here ──

# YOUR CODE HERE

---

## ✅ Knowledge Check

1. What are the 5 fields in an OHLCV candle?
2. Why does the Binance API return prices as strings instead of numbers?
3. What is the maximum number of klines per API request?
4. Why do we use `ON CONFLICT ... DO UPDATE` instead of plain `INSERT`?
5. What does `frozen=True` do in a dataclass?

<details>
<summary>Click for answers</summary>

1. Open, High, Low, Close, Volume
2. To avoid floating-point precision loss — strings preserve the exact decimal representation
3. 1000 klines per request
4. To handle duplicate fetches gracefully — if we re-fetch the same time range, existing rows are updated instead of causing an error
5. It makes instances immutable — you can't accidentally modify a kline record after creation, which prevents data corruption bugs

</details>

---

## 🔗 Next: Notebook 02 — Technical Indicators from Scratch

Now that you have raw price data, the next step is to **extract meaningful signals** from it. In Notebook 02, you'll build EMA, RSI, Bollinger Bands, MACD, and volatility from scratch using pure pandas — the exact indicators that power our trading bot's signal engines.

**Open:** `02_Technical_Indicators_from_Scratch.ipynb`